# IPF V3 — provisional current-day bridge

Run in Google Colab **after** running the QC bridge-state notebook. Paste the single `QC_BRIDGE_BUNDLE,...` line when prompted.

This does **not** replace QuantConnect as the formal V3 source. It only determines whether to provisionally arm the read-only IG quote capture while QC is one session behind.

In [ ]:
# IPF V3 provisional current-day bridge checker
# Formal V3 inclusion remains QuantConnect-authoritative.
# This notebook only decides whether to provisionally arm the read-only IG capture.

!pip -q install yfinance

import json
import math
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import yfinance as yf

P90 = 1.7796
P95 = 2.3646
MAX_OVERLAP_RETURN_DIFF_BP = 3.0

line = input("Paste the full QC_BRIDGE_BUNDLE line here:\n").strip()
if not line.startswith("QC_BRIDGE_BUNDLE,"):
    raise RuntimeError("Expected a line beginning QC_BRIDGE_BUNDLE,")

bundle = json.loads(line.split(",", 1)[1])
asof = pd.Timestamp(bundle["asof_date"])
prev_close = float(bundle["prev_close"])
prior = np.asarray(bundle["prior_returns"], dtype=float)

if len(prior) != 252:
    raise RuntimeError(f"Expected 252 prior returns, got {len(prior)}")

# Recent Yahoo minute data are used only for the session(s) QC has not yet published.
# We also use the overlap dates to deterministically align Yahoo's minute labels to QC.
raw = yf.download(
    "QQQ",
    period="8d",
    interval="1m",
    auto_adjust=False,
    progress=False,
    prepost=False,
    threads=False,
)

if raw is None or len(raw) == 0:
    raise RuntimeError("BRIDGE_ERROR,no_external_minute_data")

# Normalize possible yfinance MultiIndex columns.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = [c[0] for c in raw.columns]
raw = raw.reset_index()
time_col = "Datetime" if "Datetime" in raw.columns else raw.columns[0]
raw[time_col] = pd.to_datetime(raw[time_col])

# Convert/localize to New York.
if getattr(raw[time_col].dt, "tz", None) is not None:
    raw["time_ny"] = raw[time_col].dt.tz_convert("America/New_York")
else:
    raw["time_ny"] = raw[time_col].dt.tz_localize("America/New_York")

raw["date"] = raw["time_ny"].dt.date
raw["hhmm"] = raw["time_ny"].dt.strftime("%H:%M")
raw["Close"] = pd.to_numeric(raw["Close"], errors="coerce")

overlap = bundle.get("overlap", [])

# Determine which Yahoo timestamp convention best matches QC's 15:45 observation.
sig_candidates = ["15:44", "15:45"]
close_candidates = ["15:59", "16:00"]

def px_for(date_str, hhmm):
    d = pd.Timestamp(date_str).date()
    x = raw[(raw["date"] == d) & (raw["hhmm"] == hhmm)]
    if x.empty:
        return None
    return float(x.iloc[-1]["Close"])

def score_candidate(hhmm, field):
    diffs = []
    for r in overlap:
        y = px_for(r["date"], hhmm)
        if y is not None:
            diffs.append(abs(y - float(r[field])))
    return (np.median(diffs) if diffs else np.inf, len(diffs))

sig_scores = {h: score_candidate(h, "px1545") for h in sig_candidates}
close_scores = {h: score_candidate(h, "close1600") for h in close_candidates}

sig_hhmm = min(sig_scores, key=lambda h: sig_scores[h][0])
close_hhmm = min(close_scores, key=lambda h: close_scores[h][0])

# Validate the feed on overlap using the chosen mapping.
ret_diffs_bp = []
matched = 0
for r in overlap:
    y1545 = px_for(r["date"], sig_hhmm)
    # For each overlap date, return uses the prior day's close. We can compare
    # directly to QC's already-computed signal_return by using the preceding
    # QC close from the overlap bundle when available.
    if y1545 is None:
        continue
    # locate prior QC close in the overlap list
    dates = [x["date"] for x in overlap]
    i = dates.index(r["date"])
    if i == 0:
        continue
    prior_qc_close = float(overlap[i-1]["close1600"])
    yret = y1545 / prior_qc_close - 1.0
    diff_bp = abs(yret - float(r["signal_return"])) * 10000.0
    ret_diffs_bp.append(diff_bp)
    matched += 1

max_diff = max(ret_diffs_bp) if ret_diffs_bp else math.inf

print(
    f"BRIDGE_FEED_CHECK,matched={matched},sig_minute={sig_hhmm},close_minute={close_hhmm},"
    f"max_overlap_return_diff_bp={max_diff:.3f}"
)

if matched < 3:
    raise RuntimeError("BRIDGE_ERROR,insufficient_overlap_for_feed_validation")
if max_diff > MAX_OVERLAP_RETURN_DIFF_BP:
    raise RuntimeError(
        f"BRIDGE_ERROR,feed_mismatch,max_overlap_return_diff_bp={max_diff:.3f},"
        f"limit={MAX_OVERLAP_RETURN_DIFF_BP:.3f}"
    )

# Find the latest completed regular session later than QC's as-of date that has a 15:45 observation.
candidate_dates = sorted({
    d for d in raw["date"]
    if pd.Timestamp(d) > asof and px_for(str(d), sig_hhmm) is not None
})

if not candidate_dates:
    print(f"BRIDGE_CHECK,NO_NEW_SESSION,qc_asof={asof.date()}")
else:
    latest = candidate_dates[-1]
    px1545 = px_for(str(latest), sig_hhmm)

    # Frozen current return uses QC's authoritative previous close for the first
    # missing session. If more than one QC session is missing, stop rather than
    # chain unconfirmed external closes.
    expected_next = latest
    missing_sessions = candidate_dates
    if len(missing_sessions) > 1:
        raise RuntimeError(
            "BRIDGE_ERROR,more_than_one_missing_qc_session;"
            "refresh the QC bridge state before using the external trigger"
        )

    ret = px1545 / prev_close - 1.0
    mu = float(np.mean(prior))
    sd = float(np.std(prior, ddof=1))
    z = (ret - mu) / sd
    abs_z = abs(z)
    signal = "UP" if ret > 0 else "DOWN"
    fade = "SELL" if ret > 0 else "BUY"
    p95 = "yes" if abs_z >= P95 else "no"
    status = "PROVISIONAL_P90_TRIGGER" if abs_z >= P90 else "PROVISIONAL_NO_SIGNAL"

    print(
        f"BRIDGE_CHECK,{status},signal_date={latest},ret_bp={ret*10000:.3f},"
        f"z={z:.6f},abs_z={abs_z:.6f},signal={signal},fade={fade},P95={p95},"
        f"qc_asof={asof.date()},external_px1545={px1545:.4f}"
    )
